In [3]:
#azims 0, 10, 20, 40, 80
#eleves, 0, 10, 40
#opposite and same sex

array_azims = list(range(-90, 91, 10))
array_elevs = list(range(-20, 41, 10))

In [9]:
azim_deltas = [-80, -40, -20, -10, 0, 10, 20, 40, 80]
elev_deltas = [-40, -10, 0, 10, 40]

azim_elev_combos = dict()

for delta in azim_deltas:
    combos_within = []
    combos_crossed = []
    for azim in array_azims:
        if azim + delta in array_azims:
            azim2 = azim + delta
            if azim * azim2 < 0:
                combos_crossed.append((azim, azim2))
            else:
                combos_within.append((azim, azim2))
    azim_elev_combos[f'azim_{delta}_within'] = combos_within
    azim_elev_combos[f'azim_{delta}_crossed'] = combos_crossed

for delta in elev_deltas:
    combos = []
    for elev in array_elevs:
        if elev + delta in array_elevs:
            elev2 = elev + delta
            combos.append((elev, elev2))
    azim_elev_combos[f'elev_{delta}'] = combos

In [4]:
import pickle

# with open('azim_elev_combos.pkl', 'wb') as f:
#     pickle.dump(azim_elev_combos, f)

In [10]:
azim_elev_combos['azim_-20_crossed']

[(10, -10)]

In [5]:
import random
from copy import deepcopy
import pickle
import pandas

In [6]:
def draw_block(combo_dict):
    """
    drawing set of 90 locations for trials
    """
    trials = []

    cross_dict = {
        80: random.choice([6, 7]),
        40: 2,
        20 : random.choice([0, 1]),
        10: 0,
        0: 0,
        -10: 0,
        -40: 2,
    }
    cross_dict[-80] = 13 - cross_dict[80]
    cross_dict[-20] = 1 - cross_dict[20]

    azim_deltas = [-80, -40, -20, -10, 0, 10, 20, 40, 80]
    elev_deltas = [-40, -10, 0, 10, 40]
    sex = ['same', 'opposite']

    dict_copy = deepcopy(combo_dict)
    for a_delta in azim_deltas:
        for e_delta in elev_deltas:
            for s in sex:
                cross = False
                if cross_dict[a_delta]:
                    cross = True
                    cross_dict[a_delta] -= 1

                if cross:
                    azims = dict_copy[f'azim_{a_delta}_crossed'].pop(random.randrange(len(dict_copy[f'azim_{a_delta}_crossed'])))
                else:
                    azims = dict_copy[f'azim_{a_delta}_within'].pop(random.randrange(len(dict_copy[f'azim_{a_delta}_within'])))

                elevs = random.sample(dict_copy[f'elev_{e_delta}'], 1)[0]
                trials.append(((azims[0], elevs[0]), (azims[1], elevs[1]), s, cross, a_delta, e_delta))
    return trials

In [7]:

manifest = pandas.read_pickle('/om2/user/imgriff/datasets/spatial_audio_pipeline/assets/human_attn_experiment_v00/full_eval_trial_manifest_new_fnames.pdpkl')

In [8]:
manifest.keys()

Index(['distractor_client_id', 'distractor_clip_dur_in_s',
       'distractor_clip_end_in_s', 'distractor_clip_start_in_s',
       'distractor_corpus', 'distractor_gender', 'distractor_gender_int',
       'distractor_split', 'distractor_split_int', 'distractor_sr',
       'distractor_src_fn', 'distractor_total_file_duration_in_s',
       'distractor_word', 'src_ix', 'client_id', 'clip_dur_in_s',
       'clip_end_in_s', 'clip_start_in_s', 'corpus', 'gender', 'gender_int',
       'split', 'split_int', 'sr', 'src_fn', 'total_file_duration_in_s',
       'word', 'cue_word', 'cue_src_ix', 'cue_client_id', 'cue_src_fn',
       'cue_clip_start_in_s', 'cue_clip_end_in_s', 'gender_cond_td',
       'cue_clip_dur_in_s'],
      dtype='object')

In [11]:
def create_experiment(manifest, num_blocks=4):
    """
    create experiment with 4 blocks
    """
    experiment = []
    azim_elev_combos = pickle.load(open('/om2/user/rphess/Auditory-Attention/azim_elev_combos.pkl', 'rb'))
    for i in range(num_blocks):
        block = draw_block(azim_elev_combos)
        random.shuffle(block)
        experiment.append(block)

    words = list(manifest.word.unique())
    random.shuffle(words)

    same_gender = 'male_male'
    opp_gender = 'male_female'
    i = 0
    array_manifest = []
    for block in experiment:
        for j, trial in enumerate(block):
            trial_word = words[i]
            i += 1

            if trial[2] == 'same':
                manifest_idx = manifest[(manifest['word'] == trial_word) & (manifest['gender_cond_td'] == f'{same_gender}')].index[0]
                true_cond = same_gender
                if same_gender == 'male_male':
                    same_gender = 'female_female'
                else:
                    same_gender = 'male_male'

            else:
                manifest_idx = manifest[(manifest['word'] == trial_word) & (manifest['gender_cond_td'] == f'{opp_gender}')].index[0]
                true_cond = opp_gender
                if opp_gender == 'male_female':
                    opp_gender = 'female_male'
                else:
                    opp_gender = 'male_female'
            trial_row = manifest.iloc[manifest_idx]
            cue_src_fn = trial_row['cue_src_fn']
            target_src_fn = trial_row['src_fn']
            distractor_src_fn = trial_row['distractor_src_fn']
            distractor_word = trial_row['distractor_word']
            block[j] = block[j] + (trial_word, true_cond, manifest_idx, distractor_word)
            array_manifest.append((trial[0], trial[1], cue_src_fn, target_src_fn, distractor_src_fn))

    experiment_data = dict()
    for i, block in enumerate(experiment):
        experiment_data[f'block_{i}'] = dict()
        block_dict = experiment_data[f'block_{i}']
        for j, trial in enumerate(block):
            block_dict[f'trial_{j}'] = {'target_loc': trial[0],
                                        'distractor_loc': trial[1],
                                        'sex_cond': trial[7],
                                        'crossed': trial[3],
                                        'azim_delta': trial[4],
                                        'elev_delta': trial[5],
                                        'target_word': trial[6],
                                        'distractor_word': trial[9]}

    return experiment_data, array_manifest

In [14]:
experiment, array_manifest = create_experiment(manifest)

with open('experiment_pilot_manifest.pkl', 'wb') as f:
    pickle.dump(array_manifest, f)

In [15]:
array_manifest

[((90, 20),
  (80, 10),
  '/om/user/imgriff/datasets/spatial_audio_pipeline/assets/human_attn_experiment_v00/cue_excerpts/massive_matthewdgonzalez.wav',
  '/om/user/imgriff/datasets/spatial_audio_pipeline/assets/human_attn_experiment_v00/target_excerpts/change_matthewdgonzalez.wav',
  '/om/user/imgriff/datasets/spatial_audio_pipeline/assets/human_attn_experiment_v00/distractor_excerpts/training_popularoutcast.wav'),
 ((-40, 10),
  (-30, 10),
  '/om/user/imgriff/datasets/spatial_audio_pipeline/assets/human_attn_experiment_v00/cue_excerpts/sent_tonyle.wav',
  '/om/user/imgriff/datasets/spatial_audio_pipeline/assets/human_attn_experiment_v00/target_excerpts/river_tonyle.wav',
  '/om/user/imgriff/datasets/spatial_audio_pipeline/assets/human_attn_experiment_v00/distractor_excerpts/field_s-whistler.wav'),
 ((-60, -10),
  (20, 0),
  '/om/user/imgriff/datasets/spatial_audio_pipeline/assets/human_attn_experiment_v00/cue_excerpts/goal_persian-poet-gal.wav',
  '/om/user/imgriff/datasets/spatial_a